In [59]:
# Alignment (including self-alignment) of strings or other sequences,
# possibly weighted by feature similarity.
import re, sys
import heapq
import numpy as np
from pprint import pprint

#import panphon
sys.path.append("/home/colin/Library/Python/string2string")
from string2string.alignment import SmithWaterman

from phonopy import strings

In [71]:
def adjust_indices(align, offset):
    """
    Adjust the indices of an alignment to account for the fact that
    the second string is a suffix of the first string.
    """
    if align.indices is None:
        return None
    span1 = align.indices[0]
    span2 = align.indices[1]
    span2 = (span2[0]+offset, span2[1]+offset)
    return (span1, span2)


def markup_self_alignment(x, align, offset):
    align_strings = align.strings
    align_indices = align.indices
    if align_strings is None or align_strings[0]=='' or align_strings[1]=='':
        return None
    if align_indices is None:
        return None
    span1, span2 = adjust_indices(align, offset)
    y = x[:span1[0]] + ['⟨'] + x[span1[0]:span1[1]] + ['⟩'] + x[span1[1]:span2[0]] + ['⟨'] + x[span2[0]:span2[1]] + ['⟩'] + x[span2[1]:]
    y = ' '.join(y)
    return y


def incremental_self_alignment(x, sep=' '):
    """
    Process each prefix of a string, finding the best alignment of
    non-overlapping substrings.
    """
    if sep!='' and sep is not None:
        x = x.split(sep)

    # Initialize the alignment object
    aligner = SmithWaterman()
    
    # Initialize an empty list to store the results
    ret = set()
    
    # Iterative over prefixes.
    for i in range(2, len(x) + 1):
        prefix = x[:i]
        # Find best alignment within a prefix.
        align_best = None
        score_best = None
        offset_best = None
        for j in range(1, len(prefix)):
            prefix_ = prefix[:j]
            _suffix = prefix[j:]
            align = aligner.get_alignment( \
                prefix_, _suffix, return_indices=True)
            score = aligner.get_alignment_score( \
                align.strings[0], align.strings[1])
            ret.add((-score, align))
            if score_best is None or score > score_best:
                score_best = score
                align_best = align
                offset_best = j
        #print(align_best, score_best)
        print(f'prefix {i}, {markup_self_alignment(x, align_best, offset_best)}, {-score_best}')

    ret = list(ret)
    heapq.heapify(ret)
    return ret

In [72]:
examples = [
    "satu-satuɲa", "kəkaseh-kəkaseh", "sə-səpet", "asal-usol",
    "llama-llama", "doggy-oggy", "piggy-wiggy", "snalnal", "snalfak"
]
x = examples[1]
x = [x[i] for i in range(len(x)) if x[i] != '-']
#x = re.sub('[-]', '', x)
print(x)
aligns = incremental_self_alignment(x, sep='')
#pprint(aligns)

['k', 'ə', 'k', 'a', 's', 'e', 'h', 'k', 'ə', 'k', 'a', 's', 'e', 'h']
prefix 2, None, -0.0
prefix 3, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 4, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 5, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 6, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 7, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 8, ⟨ k ⟩ ə ⟨ k ⟩ a s e h k ə k a s e h, -1.0
prefix 9, ⟨ k ə ⟩ k a s e h ⟨ k ə ⟩ k a s e h, -5.0
prefix 10, ⟨ k ə k ⟩ a s e h ⟨ k ə k ⟩ a s e h, -9.0
prefix 11, ⟨ k ə k a ⟩ s e h ⟨ k ə k a ⟩ s e h, -13.0
prefix 12, ⟨ k ə k a s ⟩ e h ⟨ k ə k a s ⟩ e h, -17.0
prefix 13, ⟨ k ə k a s e ⟩ h ⟨ k ə k a s e ⟩ h, -21.0
prefix 14, ⟨ k ə k a s e h ⟩ ⟨ k ə k a s e h ⟩, -25.0


In [73]:
x = "s a t u s a t u ɲ a"
y = x.split(' ')
y_ = y[:4]
_y = y[4:]
# print(y_, _y)

aligner = SmithWaterman()
align = aligner.get_alignment(y_, _y, return_indices=True, return_score_matrix=True)
print(len(align))
pprint(align)

4
Alignment(strings=('s | a | t | u', 's | a | t | u'), indices=((0, 4), (0, 4)), score=None, score_matrix=array([[0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 2., 1., 0., 0., 1.],
       [0., 0., 1., 3., 2., 1., 0.],
       [0., 0., 0., 2., 4., 3., 2.]]))
